# 36 — Normalization, Gated Feed-Forward Networks, and a Decoder Block

**Master syllabus coverage:** V2 5.9–5.12

## Why this matters

The decoder block combines communication, per-token computation, normalization, and identity paths. It is the repeating unit whose correctness and stability determine the whole model.

## Learning objectives

- Implement and compare LayerNorm and RMSNorm semantics.
- Distinguish pre-norm and post-norm residual ordering.
- Build and count GELU and SwiGLU feed-forward sublayers.
- Implement a classic pre-norm residual block.
- Verify shapes and finite gradients through the complete block.
- Test causal behavior from block inputs to outputs.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. A classic pre-norm decoder block

```text
x = x + causal_attention(layer_norm_1(x))
x = x + mlp(layer_norm_2(x))
```

Attention communicates between positions. The MLP transforms each position independently.
Residual additions preserve `[B,T,C]`. Pre-norm gives both sublayers standardized inputs
and maintains a direct residual route through depth.


In [ ]:
import math
import torch
import torch.nn.functional as F

class CausalSelfAttention(torch.nn.Module):
    def __init__(self, width, heads, dropout=0.0):
        super().__init__()
        assert width % heads == 0
        self.heads, self.head_dim = heads, width // heads
        self.qkv = torch.nn.Linear(width, 3 * width)
        self.out = torch.nn.Linear(width, width)
        self.dropout = dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        def split(z): return z.view(B, T, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = map(split, (q, k, v))
        values = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=self.dropout if self.training else 0.0,
        )
        values = values.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(values)


## 2. LayerNorm, RMSNorm, pre-norm, and post-norm

LayerNorm subtracts the feature mean and divides by feature standard deviation, then
learns scale and shift. RMSNorm divides by root mean square and usually learns only scale.
Both normalize each token independently across $C$.

Pre-norm uses `x + sublayer(norm(x))`; post-norm uses `norm(x + sublayer(x))`. Their
gradient paths and checkpoint layouts differ, so the choice is part of architecture.


In [ ]:
class RMSNorm(torch.nn.Module):
    def __init__(self, width, eps=1e-6):
        super().__init__()
        self.scale = torch.nn.Parameter(torch.ones(width))
        self.eps = eps

    def forward(self, x):
        rms = x.square().mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return self.scale * (x / rms)

norm_input = torch.randn(2, 5, 32) * 3 + 4
rms_norm = RMSNorm(32)
rms_output = rms_norm(norm_input)
torch.testing.assert_close(
    rms_output.square().mean(dim=-1), torch.ones(2, 5), atol=2e-5, rtol=2e-5
)
print("LayerNorm parameters:", sum(p.numel() for p in torch.nn.LayerNorm(32).parameters()))
print("RMSNorm parameters:", sum(p.numel() for p in rms_norm.parameters()))


## 3. Position-wise GELU and SwiGLU feed-forward networks

A classic GPT-style FFN expands $C\to4C$, applies GELU, then projects $4C\to C$.
It shares weights across positions but processes each position separately. Its parameter
count is approximately $8C^2$, often more than the attention projections' $4C^2$.

Modern decoders often use SwiGLU: two input projections produce a gate and value, SiLU
gates one branch, their elementwise product is projected back to $C$. A hidden width near
$8C/3$ gives a parameter count comparable to the classic 4C FFN.


In [ ]:
class FeedForward(torch.nn.Module):
    def __init__(self, width, multiplier=4, dropout=0.0):
        super().__init__()
        hidden = multiplier * width
        self.net = torch.nn.Sequential(
            torch.nn.Linear(width, hidden),
            torch.nn.GELU(),
            torch.nn.Linear(hidden, width),
            torch.nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

width = 32
ffn = FeedForward(width)

class SwiGLU(torch.nn.Module):
    def __init__(self, width, hidden=None):
        super().__init__()
        hidden = hidden or round(8 * width / 3)
        self.gate = torch.nn.Linear(width, hidden, bias=False)
        self.value = torch.nn.Linear(width, hidden, bias=False)
        self.down = torch.nn.Linear(hidden, width, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.value(x))

swiglu = SwiGLU(width)
sample = torch.randn(2, 5, width)
assert ffn(sample).shape == swiglu(sample).shape == sample.shape
print("GELU FFN parameters:", sum(p.numel() for p in ffn.parameters()))
print("SwiGLU parameters:", sum(p.numel() for p in swiglu.parameters()))


## 4. Assemble and inspect the block

Each LayerNorm has independent learned scale and bias. Do not reuse one norm unless that
sharing is an intentional architecture decision. Dropout belongs inside update branches,
before their residual addition.


In [ ]:
class DecoderBlock(torch.nn.Module):
    def __init__(self, width, heads, dropout=0.0):
        super().__init__()
        self.norm1 = torch.nn.LayerNorm(width)
        self.attention = CausalSelfAttention(width, heads, dropout)
        self.norm2 = torch.nn.LayerNorm(width)
        self.ffn = FeedForward(width, dropout=dropout)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

torch.manual_seed(42)
block = DecoderBlock(width=32, heads=4, dropout=0.1)
x = torch.randn(2, 8, 32, requires_grad=True)
y = block(x)
assert y.shape == x.shape
y.square().mean().backward()
assert all(parameter.grad is not None and torch.isfinite(parameter.grad).all()
           for parameter in block.parameters())
print("block output:", y.shape, "parameters:", sum(p.numel() for p in block.parameters()))


## 5. Causality is an end-to-end property

A causal mask inside attention should guarantee that changing future tokens does not
change earlier block outputs in evaluation mode. This black-box test catches mask,
transpose, and accidental time-mixing bugs without trusting internal attention weights.


In [ ]:
block.eval()
prefix = torch.randn(1, 4, 32)
future_a = torch.randn(1, 3, 32)
future_b = torch.randn(1, 3, 32) * 100
sequence_a = torch.cat((prefix, future_a), dim=1)
sequence_b = torch.cat((prefix, future_b), dim=1)
with torch.inference_mode():
    output_a = block(sequence_a)
    output_b = block(sequence_b)
torch.testing.assert_close(output_a[:, :4], output_b[:, :4], atol=1e-5, rtol=1e-5)
assert not torch.allclose(output_a[:, 4:], output_b[:, 4:])
print("future perturbation cannot change prefix outputs")


## Failure modes to recognize

- **Post/pre-norm confusion:** checkpoint architecture differs despite matching names.
- **Shared normalization by accident:** attention and MLP updates use tied scale/shift parameters.
- **Wrong normalization axis:** tokens or batch items influence each other's statistics.
- **Dropout after residual sum:** the identity path is corrupted.
- **Internal-only causality check:** a later time-mixing operation reintroduces leakage.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Manually derive the block parameter count as a function of C and expansion multiplier.
2. Replace LayerNorm/GELU with RMSNorm/SwiGLU while preserving the public block contract.
3. Compare input-gradient norms across 12 pre-norm and post-norm blocks.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can compare normalization and FFN variants, assemble a block, account for every parameter, and prove finite gradients and end-to-end causality.

**Next:** 37 — Decoder stack and language-model head.
